In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_1")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}
n = 3
condition_pal = {
    "wt": blues[n],
    "bcd": reds[2],
    "trk": greens[2],
}

In [ ]:
ap_vals = np.linspace(0.001, 0.98, 6)
base_color = r"#FCF7EE"
clone_colors = ["#49643F", "#CE6750", "#98BB77"]

def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255

In [ ]:
all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

all_surface_areas = dnt.calculate_density.calculate_all_surface_areas(spots_dfs, stems, all_mmfs, ap_vals)
cycle_relative_densities = dnt.calculate_density.calculate_relative_densities(spots_dfs, stems, all_mmfs, all_surface_areas, condition_map, cycles)

## Fig 1b/c blender

In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

df = spots_dfs[0]

mesh_path = save_path / "all_meshes"
mesh_path.mkdir(exist_ok=True)

colors = [hex2rgb(c) for c in clone_colors]

track_id_colors = {}

df["initial_position"] = df.groupby("track_id")["AP"].transform("first")
df["displacement_from_start"] = (df["AP"] - df["initial_position"]) * 100
df["initial_cycle"] = df.groupby("track_id")["cycle"].transform("first")

df = df[df["frame"] == all_mmfs[stems[0]][-1]].copy()
df = df.query("initial_cycle == 10")
t = df.groupby("track_id")[["AP", "displacement_from_start", "initial_cycle"]].mean().reset_index()
print(t.index)

track_id_colors = {}
frame_df = df.query("frame == frame.min()")
frame_positions = frame_df.groupby("track_id")[["AP"]].mean()

# import napari
# viewer = napari.Viewer()
# frame_df2 = df.query("frame == frame.max()")
# labels = frame_df2["track_id"].values
# points = frame_df2[["z", "y", "x"]].values
# colors = [cc.glasbey_cool[tid % 255] for tid in frame_df2["track_id"]]
# viewer.add_points(points, properties={"tid": labels}, face_color=colors, size=10)
# napari.run()

for clone, color in zip([201, 262, 133], colors):
    track_id_colors[clone] = color

df = spots_dfs[0]

for i, frame in tqdm(enumerate(df["frame"].unique())):

    frame_df = df.query("frame == @frame")

    points = frame_df[["z", "y", "x"]].values
    mesh = dnt.mesh_from_points(points)

    mesh.write_obj(mesh_path / f"frame_{frame}.obj")
    track_ids = frame_df["track_id"].values

    colors = [track_id_colors.get(tid, hex2rgb(base_color)) for tid in track_ids]

    valid = pd.Series(np.arange(len(points))).isin(np.unique(mesh.faces))
    blender_save = pd.DataFrame(np.array(colors)[valid], columns=["R", "G", "B"])
    blender_save["track_id"] = track_ids[valid]

    blender_save.to_csv(mesh_path / f"frame_{frame}_colors.csv", index=False)

### Fig 1d barplot

In [ ]:
def get_plotting_df(relative_densities_df):
    # remove most anterior and posterior positions (pole cells)
    plotting_df = relative_densities_df.query("positions > 0.01 and positions < 0.95").copy()

    return plotting_df


def plot_compare_cycles_at_condition(relative_densities_df, condition, cycles, pal, legend=False, style="-"):
    plotting_df = get_plotting_df(relative_densities_df)

    filtered_df = plotting_df.query("condition == @condition and cycle in @cycles")

    sns.barplot(filtered_df, x="positions", y="densities", hue="cycle", palette=pal, lw=1, legend=legend, ax=ax, linestyle=style, errorbar=None, edgecolor="k", alpha=1.0)
    sns.stripplot(filtered_df, x="positions", y="densities", hue="cycle", palette="dark:k", legend=False, dodge=True, ax=ax)

    xtick_positions = ax.get_xticks()
    ax.set_xticks(xtick_positions, ["Anterior", "", "Middle", "", "Posterior"])

fig, ax = plt.subplots(1, 1, figsize=(3.5, 2.7))
cycles = [10, 14]
condition = "wt"
cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
plot_compare_cycles_at_condition(cycle_relative_densities, condition, cycles, {10: "#A2ABB9", 14: "#607380"}, legend=False)
plt.xlabel("")
plt.ylabel("")
plt.ylim(0.5, 1.25)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
